# Phase 2: Data Integration and Post-Join Cleaning
**By:** Mohamed Issa (authored the integration and the post-join cleaning)

**What this notebook does:** joins the collisions ZIP x year table and the housing ZIP x year table, cleans the joined table, validates it, and writes the one table the website reads: `data/zip_year.csv`.

| | File | Written by |
|---|---|---|
| Input 1 | `data/collisions_zip_year.csv.gz` | Yassin, phase 2 collisions ZIP x year notebook |
| Input 2 | `data/housing_zip_year.csv.gz` | Walid, `phase2_housing_zip_year_walid.ipynb` |
| Output | `data/zip_year.csv` | this notebook |

**Scope:** 2022-01-01 to 2026-10-31, the common team scope. 2026 is a partial year (January to October), so `is_partial_year` is true for 2026 rows and 2026 totals must not be compared with full years.

**How to run:** open the repository folder in VS Code, select the `.venv` kernel (installed from `requirements.txt`), then Restart Kernel and Run All. Both input files must exist in `data/`; if one is missing, the first code cell stops with an error naming the notebook to run. This notebook makes no API calls.

**Wording:** every result here is a correlation between counts in the same ZIP code and year. It does not show that construction causes collisions or the reverse.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# All paths are relative to the repository root (VS Code starts the kernel there).
assert Path("src/data.py").is_file(), "Start the kernel in the repository root (src/data.py not found)."
DATA_DIR = Path("data")
OUTPUT = DATA_DIR / "zip_year.csv"

KEYS = ["zip_code", "year"]
YEAR_MIN, YEAR_MAX = 2022, 2026
PARTIAL_YEAR = 2026

# The agreed file name comes first. Yassin's open PR currently writes the uncompressed .csv,
# so that name is accepted as a fallback; the cell prints which file was used.
INPUTS = {
    "collisions": {
        "paths": [DATA_DIR / "collisions_zip_year.csv.gz", DATA_DIR / "collisions_zip_year.csv"],
        "notebook": "Yassin's phase 2 collisions ZIP x year notebook (phase2_collisions_aggregation_yassin.ipynb)",
        "columns": ["zip_code", "year", "borough", "collision_count", "persons_injured", "persons_killed",
                    "pedestrians_injured", "pedestrians_killed", "cyclist_injured", "cyclist_killed",
                    "motorist_injured", "motorist_killed", "is_partial_year"],
    },
    "housing": {
        "paths": [DATA_DIR / "housing_zip_year.csv.gz", DATA_DIR / "housing_zip_year.csv"],
        "notebook": "Walid's phase2_housing_zip_year_walid.ipynb",
        "columns": ["zip_code", "year", "borough", "housing_job_count", "new_building_count",
                    "alteration_job_count", "demolition_job_count",
                    "proposed_dwelling_units_new_buildings", "is_partial_year"],
    },
}


def parse_bool(series, name):
    """Return a bool column from True/False values stored as bool or text; stop on anything else."""
    if series.dtype == bool:
        return series
    text = series.astype("string").str.strip().str.lower()
    bad = sorted(set(text.dropna()) - {"true", "false"})
    if bad or text.isna().any():
        raise ValueError(f"{name}: expected only True/False, found {bad[:5]} and {int(text.isna().sum())} missing")
    return text.eq("true").astype(bool)


def load_input(name):
    """Read one ZIP x year table with zip_code and borough as text and year as an integer."""
    spec = INPUTS[name]
    path = next((p for p in spec["paths"] if p.exists()), None)
    if path is None:
        raise FileNotFoundError(
            f"{spec['paths'][0]} not found. Run {spec['notebook']} first; it writes this file.")
    df = pd.read_csv(path, dtype={"zip_code": "string", "borough": "string"},
                     keep_default_na=False, na_values=[""])
    missing = [c for c in spec["columns"] if c not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing the agreed columns {missing}")
    df["year"] = df["year"].astype("int64")
    df["is_partial_year"] = parse_bool(df["is_partial_year"], f"{name}.is_partial_year")
    print(f"{name}: read {path} | {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df


collisions = load_input("collisions")
housing = load_input("housing")

## Input checks

Before joining, each input has to follow the agreed interface (`docs/CONTRACT.md`): 5-digit text ZIP codes, integer years from 2022 to 2026, exactly one row per (`zip_code`, `year`), and non-negative integer counts with no missing values. The next cell checks each rule on both inputs, prints PASS or FAIL, and stops if anything fails. It also records each input's totals, which the reconciliation at the end compares against.

In [ ]:
def count_columns(df):
    """Integer count columns of an input: every integer column except year."""
    return [c for c in df.columns if c != "year" and pd.api.types.is_integer_dtype(df[c])]


COLLISION_COUNTS = count_columns(collisions)
HOUSING_COUNTS = count_columns(housing)
assert not set(COLLISION_COUNTS) & set(HOUSING_COUNTS), "the two inputs share a count column name"


def check_input(name, df, counts):
    checks = {
        "zip_code is a 5-digit string": df["zip_code"].notna().all() and df["zip_code"].str.fullmatch(r"\d{5}").all(),
        f"year between {YEAR_MIN} and {YEAR_MAX}": df["year"].between(YEAR_MIN, YEAR_MAX).all(),
        "one row per (zip_code, year)": not df.duplicated(KEYS).any(),
        "counts are integers with no missing values": all(pd.api.types.is_integer_dtype(df[c]) for c in counts),
        "counts are non-negative": (df[counts] >= 0).all().all(),
        f"is_partial_year is true exactly for {PARTIAL_YEAR}": df["is_partial_year"].eq(df["year"].eq(PARTIAL_YEAR)).all(),
    }
    for rule, ok in checks.items():
        print(f"{'PASS' if ok else 'FAIL'}  {name}: {rule}")
    assert all(checks.values()), f"{name} input does not follow the agreed interface (see FAIL lines)"


check_input("collisions", collisions, COLLISION_COUNTS)
check_input("housing", housing, HOUSING_COUNTS)

INPUT_TOTALS = {
    "collision_count": int(collisions["collision_count"].sum()),
    "persons_injured": int(collisions["persons_injured"].sum()),
    "housing_job_count": int(housing["housing_job_count"].sum()),
    "new_building_count": int(housing["new_building_count"].sum()),
}
print()
print("Count columns, collisions:", COLLISION_COUNTS)
print("Count columns, housing:   ", HOUSING_COUNTS)
print("Extra non-count columns:  ", [c for c in list(collisions.columns) + list(housing.columns)
                                     if c not in KEYS + ["borough", "is_partial_year"] + COLLISION_COUNTS + HOUSING_COUNTS])
print()
years = pd.DataFrame({
    "collisions ZIP-years": collisions.groupby("year").size(),
    "housing ZIP-years": housing.groupby("year").size(),
    "collision_count": collisions.groupby("year")["collision_count"].sum(),
    "housing_job_count": housing.groupby("year")["housing_job_count"].sum(),
}).fillna(0).astype("int64")
print(years.to_string())
print()
for k, v in INPUT_TOTALS.items():
    print(f"Input total {k}: {v:,}")

## Join key and granularity

**Key: (`zip_code`, `year`).** The two datasets share no row-level key. A crash record and a building filing are different events, and there is no ID that links one crash to one building. Matching individual crashes to nearby buildings (rejected alternative) would need a distance rule that nobody agreed, would leave out crashes without coordinates, and would not give the ZIP x year table the team agreed on (`docs/CONTRACT.md` 10.2). So both sides were first aggregated to the same grain (one row per ZIP code and year, done in Yassin's and Walid's notebooks), and this notebook joins those two summary tables. A crash is never matched to one building; a row only says "in this ZIP and this year there were N collisions and M housing jobs".

**Join type: outer.** An inner join keeps only ZIP-years that appear in both tables, so it would silently drop ZIP-years with crashes but no housing filings (and the reverse). Those are real observations and matter for the comparison, so the outer join keeps every ZIP-year from either side. The `validate="one_to_one"` argument makes pandas stop if either side has a repeated key, so the join cannot multiply rows. The cell then checks that the joined table has exactly one row per key in the union of both inputs.

In [ ]:
coll = collisions.rename(columns={"borough": "borough_collisions", "is_partial_year": "partial_collisions"})
hous = housing.rename(columns={"borough": "borough_housing", "is_partial_year": "partial_housing"})
shared = (set(coll.columns) & set(hous.columns)) - set(KEYS)
assert not shared, f"columns on both sides besides the keys: {shared}"

joined = coll.merge(hous, on=KEYS, how="outer", validate="one_to_one", indicator="match_status")
joined["match_status"] = joined["match_status"].map(
    {"both": "both", "left_only": "collisions only", "right_only": "housing only"}).astype("string")

union_keys = pd.concat([collisions[KEYS], housing[KEYS]]).drop_duplicates()
assert len(joined) == len(union_keys), "the join changed the number of distinct ZIP-years"
assert not joined.duplicated(KEYS).any()

print("Collisions ZIP-years:", f"{len(collisions):,}")
print("Housing ZIP-years:   ", f"{len(housing):,}")
print("Joined ZIP-years:    ", f"{len(joined):,}", "(equals the number of distinct keys in both inputs)")
print()
print(joined["match_status"].value_counts().reindex(["both", "collisions only", "housing only"]).to_string())
print()
print(pd.crosstab(joined["year"], joined["match_status"]).to_string())

## Filling the side that has no row

After the outer join, a "collisions only" row has no housing counts and a "housing only" row has no collision counts. `docs/CONTRACT.md` (section 10.5) says not to fill every unmatched value with zero automatically, so here is why zero is the right value for these columns:

- Yassin's and Walid's tables only contain ZIP-years where at least one event was recorded. A ZIP-year missing from the housing table means no `I1` filing was recorded for that ZIP in that year. That is a count of zero, not unknown data.
- Likewise, a ZIP-year missing from the collisions table means no crash with that ZIP code was recorded in that year.

So **every integer count column is filled with 0**. The cell prints how many values were filled per column. One limit stays: crashes whose ZIP code was missing in the source are not in any ZIP-year at all (Yassin's notebook reports how many, per year). Their absence is a data-quality limitation, not a zero, and this table cannot recover them.

**Borough:** one borough per ZIP code, so every year of a ZIP shows the same borough on the website. The collisions value is used when the collisions table has the ZIP (Ahmed's pipeline verified those boroughs), otherwise the housing value. Every case where the two sides disagree is printed. Values are Title Case.

**`is_partial_year`** is recomputed as `year == 2026` for every row, including rows that came from only one side.

In [ ]:
COUNT_COLS = COLLISION_COUNTS + HOUSING_COUNTS
filled = joined[COUNT_COLS].isna().sum()
joined[COUNT_COLS] = joined[COUNT_COLS].fillna(0).astype("int64")
print("Values filled with 0, per count column:")
print(filled.to_string())
print()

# One borough per ZIP: collisions value first, housing value as the fallback.
def zip_borough(df, col):
    """Most frequent non-missing borough per ZIP (ties: alphabetical)."""
    b = df.dropna(subset=[col]).groupby(["zip_code", col]).size().rename("n").reset_index()
    b = b.sort_values(["zip_code", "n", col], ascending=[True, False, True])
    return b.drop_duplicates("zip_code").set_index("zip_code")[col]

borough_c = zip_borough(joined, "borough_collisions")
borough_h = zip_borough(joined, "borough_housing")
both_sides = borough_c.index.intersection(borough_h.index)
conflicts = pd.DataFrame({"collisions": borough_c[both_sides], "housing": borough_h[both_sides]})
conflicts = conflicts[conflicts["collisions"] != conflicts["housing"]]
print("ZIPs whose borough differs between the two inputs:", len(conflicts))
if len(conflicts):
    totals = joined.groupby("zip_code")[["collision_count", "housing_job_count"]].sum()
    print(conflicts.join(totals).to_string())
print()

zip_to_borough = borough_c.combine_first(borough_h).str.strip().str.title()
joined["borough"] = joined["zip_code"].map(zip_to_borough).astype("string")
joined["is_partial_year"] = joined["year"].eq(PARTIAL_YEAR)

print("Rows with no borough on either side:", int(joined["borough"].isna().sum()))
print("Borough values:", sorted(joined["borough"].dropna().unique()))
print("Rows where an input's is_partial_year disagrees with year == 2026:",
      int((joined["partial_collisions"].notna() & joined["partial_collisions"].ne(joined["is_partial_year"])).sum()
          + (joined["partial_housing"].notna() & joined["partial_housing"].ne(joined["is_partial_year"])).sum()))
joined = joined.drop(columns=["borough_collisions", "borough_housing", "partial_collisions", "partial_housing"])

## Post-join cleaning 1: ZIP codes that are not NYC ZIP codes

Both inputs already contain only 5-digit ZIP codes, but a 5-digit number is not automatically an NYC ZIP code (`docs/CONTRACT.md` 5.1). NYC ZIP codes start with these three digits: 100, 101, 102 (Manhattan), 103 (Staten Island), 104 (Bronx), 112 (Brooklyn), and 110, 111, 113, 114, 116 (Queens).

**Rule:**
- **Drop** a ZIP code that does not start with one of those prefixes. It cannot be placed on the NYC map and is almost certainly a typing error in the source.
- **Drop** a placeholder: `00000`, `99999`, `10000` and `11111` are not ZIP codes assigned by USPS. They look like values typed in when the real ZIP was unknown.
- **Drop** a ZIP code that passes the prefix test but is known to be outside NYC. The prefix test is only a first filter: the `110` range also covers Nassau County, so `11021` (Great Neck, Nassau County) is listed by name.
- **Keep** every other NYC ZIP code, even when it appears in only one input or has very small counts. Many of these are real non-residential ZIP codes: Manhattan has ZIP codes for single office buildings, and there are ZIP codes for airports and parks. Small counts there are real observations. Removing them would change the totals for no reason; they are flagged with `low_data_flag` in the next step instead.
- A real residential ZIP code that has housing jobs in every year but no collisions at all is also kept: its zero comes from the collisions table, where a crash's ZIP code depends on how its location was coded. It is printed in the table below so it can be checked, and it is listed as a limitation.
- **Drop** a row that has no borough after the join. The website needs a borough on every row, and a borough must not be invented (`docs/CONTRACT.md` 4.3).

**Alternative rejected:** checking every ZIP against an official list of NYC ZIP codes. That needs another dataset that is not part of the project. The prefix rule, plus the named exceptions found by looking at the table below, covers the clear errors without one.

The first table lists every ZIP code that appears in only one input, with its totals across all years, so the decision can be checked by eye.

In [ ]:
NYC_PREFIXES = ("100", "101", "102", "103", "104", "110", "111", "112", "113", "114", "116")
PLACEHOLDERS = {"00000", "99999", "10000", "11111"}    # not USPS-assigned ZIP codes
NON_NYC = {"11021": "Great Neck, Nassau County"}       # passes the prefix test but is outside NYC

per_zip = joined.groupby("zip_code").agg(
    borough=("borough", "first"),
    years=("year", "nunique"),
    collision_count=("collision_count", "sum"),
    housing_job_count=("housing_job_count", "sum"),
    in_collisions=("match_status", lambda s: s.isin(["both", "collisions only"]).any()),
    in_housing=("match_status", lambda s: s.isin(["both", "housing only"]).any()),
)
one_side = per_zip[per_zip["in_collisions"] != per_zip["in_housing"]]
print("ZIP codes in the joined table:", len(per_zip))
print("ZIP codes that appear in only one input:", len(one_side))
print(one_side.sort_values(["collision_count", "housing_job_count"]).to_string())
print()

non_nyc = (~joined["zip_code"].str.startswith(NYC_PREFIXES) | joined["zip_code"].isin(PLACEHOLDERS)
           | joined["zip_code"].isin(list(NON_NYC)))
no_borough = joined["borough"].isna()
drop = non_nyc | no_borough
dropped = joined[drop]
print("ZIP codes dropped by the rule:", sorted(joined.loc[non_nyc, "zip_code"].unique()))
print("Rows dropped, non-NYC or placeholder ZIP:", int(non_nyc.sum()))
print("Rows dropped, no borough:", int((no_borough & ~non_nyc).sum()))
if len(dropped):
    print(dropped[KEYS + ["borough", "match_status", "collision_count", "housing_job_count"]].to_string())
DROPPED_TOTALS = {k: int(dropped[k].sum()) for k in INPUT_TOTALS}

rows_before = len(joined)
zip_year = joined[~drop].reset_index(drop=True)
print()
print("Rows before:", f"{rows_before:,}", "| dropped:", int(drop.sum()), "| rows left:", f"{len(zip_year):,}")

## Post-join cleaning 2: outliers and low-data rows

**Rule: flag, do not delete.** Large counts are expected in dense or fast-growing ZIP codes, so a large value is not an error by itself (`docs/CONTRACT.md` 7: do not remove valid extreme observations). Two flags are added instead:

- `outlier_flag`: true when `collision_count`, `housing_job_count` or `new_building_count` is above the upper IQR fence (Q3 + 1.5 x IQR) **within its year**. Fences are per year because 2026 is a partial year and would otherwise look low next to full years. The cell prints the fences and the flagged rows with the largest values, so they can be checked.
- `low_data_flag`: true when `collision_count < 5` **and** `housing_job_count < 5`. Ratios and ranks on such rows depend on one or two events, so charts can leave them out or mark them.

**Domain checks.** These rules must always hold, so the cell stops if one fails: no negative counts, `new_building_count <= housing_job_count`, and `persons_killed` and `persons_injured` are zero wherever `collision_count` is zero. The cell also reports, without changing anything, how many rows have `pedestrians_injured + cyclist_injured + motorist_injured` different from `persons_injured`. The source reports these fields separately, and they do not always add up.

In [ ]:
IQR_COLUMNS = ["collision_count", "housing_job_count", "new_building_count"]

q = zip_year.groupby("year")[IQR_COLUMNS].quantile([0.25, 0.75]).unstack()
fences = pd.DataFrame({c: q[(c, 0.75)] + 1.5 * (q[(c, 0.75)] - q[(c, 0.25)]) for c in IQR_COLUMNS})
print("Upper IQR fences per year:")
print(fences.round(1).to_string())
print()

above = pd.DataFrame({c: zip_year[c] > zip_year["year"].map(fences[c]) for c in IQR_COLUMNS})
zip_year["outlier_flag"] = above.any(axis=1)
zip_year["low_data_flag"] = zip_year["collision_count"].lt(5) & zip_year["housing_job_count"].lt(5)
for c in IQR_COLUMNS:
    print(f"Rows above the fence for {c}: {int(above[c].sum())}")
print("Rows with outlier_flag:", int(zip_year["outlier_flag"].sum()))
print("Rows with low_data_flag:", int(zip_year["low_data_flag"].sum()))
print()
print("Flagged rows with the largest collision_count:")
print(zip_year[zip_year["outlier_flag"]].nlargest(5, "collision_count")[KEYS + ["borough"] + IQR_COLUMNS].to_string(index=False))
print("Flagged rows with the largest housing_job_count:")
print(zip_year[zip_year["outlier_flag"]].nlargest(5, "housing_job_count")[KEYS + ["borough"] + IQR_COLUMNS].to_string(index=False))
print()

domain = {
    "no negative counts": (zip_year[COUNT_COLS] >= 0).all().all(),
    "new_building_count <= housing_job_count": (zip_year["new_building_count"] <= zip_year["housing_job_count"]).all(),
    "no injuries or deaths where collision_count is 0":
        (zip_year.loc[zip_year["collision_count"].eq(0), ["persons_injured", "persons_killed"]] == 0).all().all(),
}
for rule, ok in domain.items():
    print(f"{'PASS' if ok else 'FAIL'}  {rule}")
assert all(domain.values()), "a domain rule failed (see FAIL lines)"
parts = zip_year[["pedestrians_injured", "cyclist_injured", "motorist_injured"]].sum(axis=1)
print("Rows where pedestrian + cyclist + motorist injured differs from persons_injured (reported, not changed):",
      int(parts.ne(zip_year["persons_injured"]).sum()))

## Post-join cleaning 3: re-check the table

The same rules as the input checks, now on the joined table: no missing values in any count column, integer counts, one row per (`zip_code`, `year`), 5-digit text ZIP codes, years 2022 to 2026, a Title Case borough on every row, and `is_partial_year` true exactly for 2026.

In [ ]:
BOROUGHS = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
rechecks = {
    "no missing values in count columns": not zip_year[COUNT_COLS].isna().any().any(),
    "count columns are integers": all(pd.api.types.is_integer_dtype(zip_year[c]) for c in COUNT_COLS),
    "one row per (zip_code, year)": not zip_year.duplicated(KEYS).any(),
    "zip_code is a 5-digit string": zip_year["zip_code"].str.fullmatch(r"\d{5}").all(),
    "year between 2022 and 2026": zip_year["year"].between(YEAR_MIN, YEAR_MAX).all(),
    "borough is one of the five, no blanks": zip_year["borough"].isin(BOROUGHS).all(),
    "is_partial_year is true exactly for 2026": zip_year["is_partial_year"].eq(zip_year["year"].eq(PARTIAL_YEAR)).all(),
}
for rule, ok in rechecks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {rule}")
assert all(rechecks.values()), "the joined table failed a re-check (see FAIL lines)"
print("Shape:", zip_year.shape)

## Derived columns for the story

- **`new_buildings_per_year_rank`**: rank of `new_building_count` among all ZIP codes in the same year, `1` = most new buildings. Formula: `zip_year.groupby("year")["new_building_count"].rank(method="min", ascending=False)`. Tied ZIP codes share the best rank (`method="min"`), so two ZIP codes tied for first are both `1` and the next one is `3`.
- **`injuries_per_100_collisions`**: `persons_injured / collision_count * 100`, rounded to 2 decimals, and only where `collision_count > 0`. Where `collision_count` is 0 the value is left empty (NaN) instead of dividing by zero. It is not filled with 0, because "no crashes" is not the same as "crashes without injuries". It describes how severe the recorded crashes were, not how dangerous a ZIP code is: there is no traffic-volume denominator.

In [ ]:
zip_year["new_buildings_per_year_rank"] = (
    zip_year.groupby("year")["new_building_count"].rank(method="min", ascending=False).astype("int64"))
has_crashes = zip_year["collision_count"] > 0
zip_year["injuries_per_100_collisions"] = np.where(
    has_crashes, (zip_year["persons_injured"] / zip_year["collision_count"].where(has_crashes) * 100).round(2), np.nan)

print("Rank 1 per year:")
print(zip_year[zip_year["new_buildings_per_year_rank"].eq(1)][["year", "zip_code", "borough", "new_building_count"]]
      .sort_values("year").to_string(index=False))
print()
print("Rows with injuries_per_100_collisions left empty (collision_count = 0):",
      int(zip_year["injuries_per_100_collisions"].isna().sum()))
print(zip_year["injuries_per_100_collisions"].describe().round(2).to_string())

## Export and the website contract

`data/zip_year.csv` starts with the 12 columns the website requires, in this order: `zip_code, year, borough, housing_job_count, new_building_count, collision_count, persons_injured, persons_killed, pedestrians_injured, cyclist_injured, motorist_injured, is_partial_year`. The other input columns, the two flags, `match_status` and the derived columns follow them; the website allows extra columns and ignores them.

The cell writes the file, reloads it with `zip_code` as text, checks that the reload equals the table in memory, and runs the website's own validator (`src.data.validate`), first in this kernel and then with the exact command from the task, in a separate Python process.

In [ ]:
REQUIRED = ["zip_code", "year", "borough", "housing_job_count", "new_building_count", "collision_count",
            "persons_injured", "persons_killed", "pedestrians_injured", "cyclist_injured", "motorist_injured",
            "is_partial_year"]
DERIVED = ["match_status", "low_data_flag", "outlier_flag", "new_buildings_per_year_rank", "injuries_per_100_collisions"]
extras = [c for c in zip_year.columns if c not in REQUIRED + DERIVED]
zip_year = zip_year[REQUIRED + extras + DERIVED].sort_values(KEYS).reset_index(drop=True)

zip_year.to_csv(OUTPUT, index=False)
print("Saved", OUTPUT, zip_year.shape)
print("Columns:", list(zip_year.columns))

reloaded = pd.read_csv(OUTPUT, dtype={"zip_code": str})
pd.testing.assert_frame_equal(reloaded[REQUIRED], zip_year[REQUIRED], check_dtype=False)
assert reloaded.shape == zip_year.shape
print("Reloaded shape:", reloaded.shape, "| required columns identical after reload: True")
print("zip_code after reload:", reloaded["zip_code"].dtype, "| is_partial_year after reload:", reloaded["is_partial_year"].dtype)

from src.data import validate
print("validate() in this kernel:", validate(reloaded).shape)

cmd = ("import pandas as pd; from src.data import validate; "
       "print(validate(pd.read_csv('data/zip_year.csv', dtype={'zip_code': str})).shape)")
result = subprocess.run([sys.executable, "-c", cmd], capture_output=True, text=True)
print("python -c validator command, exit code", result.returncode, "| output:", result.stdout.strip(), result.stderr.strip())
assert result.returncode == 0, "the website validator rejected data/zip_year.csv"

## Validating the integration

Three checks that the join did what it should:
1. **Coverage:** how many ZIP-years are in both inputs, in collisions only, and in housing only, per year.
2. **Totals reconcile:** the joined table must hold exactly the input totals. The join and the zero-fill add or lose nothing; the only allowed difference is what the ZIP rule above dropped, and the cell prints that.
3. **Relationship:** `housing_job_count` against `collision_count` per ZIP code, one panel per year, with the Spearman rank correlation. Spearman is used because both counts are very skewed and the relationship need not be linear; it is computed as the Pearson correlation of the ranks. Both axes are logarithmic. Since many counts are 0 and log(0) is undefined, the axes show count + 1.

In [ ]:
coverage = pd.crosstab(zip_year["year"], zip_year["match_status"]).reindex(
    columns=["both", "collisions only", "housing only"], fill_value=0)
ax = coverage.plot(kind="bar", stacked=True, figsize=(9, 4), color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("ZIP-years by input coverage (2026: January to October)")
ax.set_xlabel("Year")
ax.set_ylabel("Number of ZIP-years")
ax.legend(title="Present in", loc="upper left", bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(coverage.to_string())

In [ ]:
reconcile = pd.DataFrame({
    "input total": pd.Series(INPUT_TOTALS),
    "dropped by ZIP rule": pd.Series(DROPPED_TOTALS),
    "zip_year.csv total": zip_year[list(INPUT_TOTALS)].sum(),
})
reconcile["input - dropped"] = reconcile["input total"] - reconcile["dropped by ZIP rule"]
reconcile["matches"] = reconcile["input - dropped"].eq(reconcile["zip_year.csv total"])
print(reconcile.to_string())
assert reconcile["matches"].all(), "totals in zip_year.csv do not reconcile with the inputs"
print()
print(f"collision_count: Yassin's file {INPUT_TOTALS['collision_count']:,} | zip_year.csv {int(zip_year['collision_count'].sum()):,}")
print(f"housing_job_count: Walid's file {INPUT_TOTALS['housing_job_count']:,} | zip_year.csv {int(zip_year['housing_job_count'].sum()):,}")

In [ ]:
years_present = sorted(zip_year["year"].unique())
fig, axes = plt.subplots(1, len(years_present), figsize=(4 * len(years_present), 4), sharex=True, sharey=True)
correlations = {}
for ax, year in zip(np.atleast_1d(axes), years_present):
    d = zip_year[zip_year["year"].eq(year)]
    # Spearman rho = Pearson correlation of the ranks (ties get their average rank).
    # Written out because pandas' method="spearman" needs scipy, which is not in requirements.txt.
    rho = d["housing_job_count"].rank().corr(d["collision_count"].rank())
    correlations[year] = (rho, len(d))
    ax.scatter(d["housing_job_count"] + 1, d["collision_count"] + 1, s=12, alpha=0.5)
    ax.set_xscale("log")
    ax.set_yscale("log")
    label = f"{year} (Jan-Oct)" if year == PARTIAL_YEAR else str(year)
    ax.set_title(f"{label}\nSpearman rho = {rho:.2f}, n = {len(d)} ZIPs")
    ax.set_xlabel("Housing jobs started (I1) + 1, log scale")
np.atleast_1d(axes)[0].set_ylabel("Collisions + 1, log scale")
fig.suptitle("Housing jobs vs collisions per ZIP code: a correlation, not a cause", y=1.03)
plt.tight_layout()
plt.show()

for year, (rho, n) in correlations.items():
    print(f"{year}: Spearman rho = {rho:.3f} over {n} ZIP codes")

**Reading the scatter plots.** Each point is one ZIP code in one year. The Spearman value printed above each panel measures whether ZIP codes with more housing jobs *also tend to* have more recorded collisions. A positive value is a **correlation only**. Both counts grow with the size and activity of a ZIP code (more people, more streets, more traffic), and there is no traffic-volume or population denominator in this table. A correlation here does not show that construction causes collisions.

## Summary

The table below is printed from this run, so its numbers always match the outputs above.

In [ ]:
summary = pd.Series({
    "Collisions ZIP-years (input)": len(collisions),
    "Housing ZIP-years (input)": len(housing),
    "Joined ZIP-years (outer join)": rows_before,
    "  in both inputs": int((joined["match_status"] == "both").sum()),
    "  collisions only": int((joined["match_status"] == "collisions only").sum()),
    "  housing only": int((joined["match_status"] == "housing only").sum()),
    "Rows dropped by the ZIP rule": rows_before - len(zip_year),
    "Rows in data/zip_year.csv": len(zip_year),
    "Columns in data/zip_year.csv": zip_year.shape[1],
    "Distinct ZIP codes": zip_year["zip_code"].nunique(),
    "Rows with outlier_flag": int(zip_year["outlier_flag"].sum()),
    "Rows with low_data_flag": int(zip_year["low_data_flag"].sum()),
})
print(summary.map("{:,}".format).to_string())

## Limitations

- **Missing ZIP codes in collisions:** crashes with no ZIP code in the source cannot be placed in a ZIP-year and are not in this table. The missing share differs by year (Yassin's notebook prints it per year), so year-to-year changes in `collision_count` partly reflect data coverage.
- **Partial 2026:** 2026 covers January to October only. Compare 2026 with other years only as a partial year.
- **No denominators:** the table has no traffic volume, population or road length, so the counts are not rates and do not measure how dangerous a ZIP code is.
- **Different events:** a housing filing date and a crash date are different events. Joining by year aligns them in time and says nothing about cause.
- **Housing-only ZIP codes:** some real NYC ZIP codes have housing jobs but no ZIP-coded collisions (see the ZIP table in post-join cleaning 1). Their `collision_count` of 0 comes from how crash locations were coded, so it may understate crashes there.
- **ZIP codes are not neighbourhoods:** ZIP codes differ a lot in size and land use, and crashes are located by where they were reported.

## Handoff to the website

- **File:** `data/zip_year.csv`, read by `src/data.py` (`load_data()` picks it up automatically instead of the dummy table).
- **Grain:** one row per (`zip_code`, `year`), 2022 to 2026. The row and column counts are printed in the summary above.
- **Required columns first:** `zip_code` (5-digit text), `year` (int), `borough` (Title Case), `housing_job_count`, `new_building_count` (I1-based job counts, Walid's definitions), `collision_count`, `persons_injured`, `persons_killed`, `pedestrians_injured`, `cyclist_injured`, `motorist_injured` (int), `is_partial_year` (bool).
- **Extras:** the other input columns, `match_status`, `low_data_flag`, `outlier_flag`, `new_buildings_per_year_rank`, `injuries_per_100_collisions` (empty where `collision_count` is 0).

## Prompts used

Full log in `AI_PROMPTS.txt` (entries by Mohamed Issa).

AI-assisted cells:
- **Claude Code:** wrote this notebook from the team lead's Phase 2 task (join key and outer join, zero-fill rule, ZIP rule, flags, derived columns, export and validation). The decisions in the markdown were checked against `docs/CONTRACT.md` and the printed outputs.